# (Re-)Imag(in)ing Price Trends — unified notebook (Mac M4 · Kaggle 2×T4)

One notebook, both machines. It **auto-detects** the device and sizes the parallelism to it:

| | Mac M4 (`mps`) | Kaggle 2×T4 (`cuda`) |
|---|---|---|
| Image generation | all performance cores | (skipped — dataset uploaded) |
| Training | 1 model at a time (Metal parallelises internally) | `MODELS_PER_GPU × 2` models **concurrently**, round-robined across both GPUs |
| Loader workers | auto | auto-capped so `MAX_PARALLEL × NUM_WORKERS ≤ vCPU` |

**Resumable across machines.** Every ensemble member's test predictions are checkpointed to `PRED_DIR` as a `.npy`. Train 2 members on the Mac, copy the `.npy` files over, train the other 3 on Kaggle — the notebook assembles the ensemble from whatever it finds and skips anything already done. Config-level results are appended to a CSV that is also resumed.

**Correctness fixes baked in** (see §3):
- the blank-image indexing bug is fixed at generation *and* filtered at load, so **old, buggy datasets still work**;
- `SIGNAL_GATE=False` makes the CNN's sample set depend only on the image (paper-faithful);
- `BAGGING=False` — the paper's fixed-split ensemble;
- non-overlapping decile portfolios (JKX Table 3).

**Memory:** models, loaders and worker processes are explicitly torn down between members, and RSS + GPU memory are printed every epoch so leaks are visible, not inferred.

In [3]:
import os, gc, sys, time, json, tempfile, threading, platform
import multiprocessing as mp
try: mp.set_start_method('fork', force=True)     # fork lets render workers inherit data copy-on-write
except RuntimeError: pass
import concurrent.futures as cf
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

try:
    import psutil; _PROC = psutil.Process()
except ImportError:
    _PROC = None

# ---------------------------------------------------------------- device
ON_KAGGLE = os.path.isdir('/kaggle')
if torch.cuda.is_available():
    BACKEND = 'cuda'
    DEVICES = [f'cuda:{i}' for i in range(torch.cuda.device_count())]
    torch.backends.cudnn.benchmark = True          # fixed input shapes per config
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    BACKEND, DEVICES = 'mps', ['mps']
else:
    BACKEND, DEVICES = 'cpu', ['cpu']

CPU = mp.cpu_count()

# ---------------------------------------------------------------- knobs
BATCH_SIZE = 128            # paper
EPOCHS     = 100
LR         = 1e-5
PATIENCE   = 2
USE_AMP    = False          # False = fp32 (paper-exact). True = fp16 autocast on CUDA (~1.5-2x, slightly different numerics)

# concurrency: several whole models at once on CUDA; Metal prefers one model with internal parallelism
MODELS_PER_GPU = 1
if BACKEND == 'cuda':
    MAX_PARALLEL = max(1, len(DEVICES) * MODELS_PER_GPU)
    NUM_WORKERS  = max(0, min(2, CPU // MAX_PARALLEL))     # never oversubscribe the CPU
else:
    MAX_PARALLEL = 1
    NUM_WORKERS  = min(4, max(0, CPU - 2))

# ensemble / methodology
ENSEMBLE_SIZE     = 5
BAGGING           = False   # paper-faithful: one fixed split, diversity from random init
TRAIN_FRAC        = 0.7
SHARED_SPLIT_SEED = 42
SIGNAL_GATE       = False   # False = CNN samples gated by the IMAGE only (paper). True = also require MOM/STR/WSTR/TREND
                            # (the old, buggy behaviour). Changing this CHANGES the dataset -> regenerate.
DROP_INCOMPLETE   = True    # drop windows without a full lookback. No-op on a clean dataset; RESCUES an old buggy one.

# which configs to run in THIS session (lets you split work across machines)
IMAGE_SIZES = [5, 20, 60]
HORIZONS    = [5, 20, 60]
TESTING     = False         # True -> 3 tickers, 1 config, 2 members

# ---------------------------------------------------------------- paths
if ON_KAGGLE:
    IMG_DIR      = '/kaggle/input/price-trend-images'     # <-- EDIT: uploaded image dataset (read-only)
    PARQUET_PATH = '/kaggle/input/market-data-sp500-1997-2023-parquet/market_data_sp500_1997_2023.parquet'
    OUT_DIR      = '/kaggle/working'
else:
    IMG_DIR      = './image_store'                        # generated here on the Mac
    PARQUET_PATH = './market_data_sp500_1997_2023.parquet'
    OUT_DIR      = '.'
PRED_DIR    = os.path.join(OUT_DIR, 'preds');   os.makedirs(PRED_DIR, exist_ok=True)
RESULTS_CSV = os.path.join(OUT_DIR, 'results.csv')
os.makedirs(IMG_DIR, exist_ok=True) if not ON_KAGGLE else None

# ---------------------------------------------------------------- logging (thread-safe: concurrent models interleave)
_LOCK = threading.Lock()
_T0   = time.time()

def log(msg, tag=''):
    el = time.time() - _T0
    with _LOCK:
        print(f"[{el//3600:02.0f}:{(el%3600)//60:02.0f}:{el%60:04.1f}]{(' '+tag) if tag else ''} {msg}", flush=True)

def mem_str(dev=None):
    parts = []
    if _PROC is not None:
        parts.append(f"rss {_PROC.memory_info().rss/1e9:.1f}G")
    if BACKEND == 'cuda' and dev is not None:
        i = dev.index or 0
        parts.append(f"gpu{i} {torch.cuda.memory_allocated(i)/1e9:.1f}/{torch.cuda.memory_reserved(i)/1e9:.1f}G")
    elif BACKEND == 'mps':
        try: parts.append(f"mps {torch.mps.current_allocated_memory()/1e9:.1f}G")
        except Exception: pass
    return " ".join(parts)

def free_memory(dev=None):
    """Called between ensemble members and configs. The single most important anti-leak step."""
    gc.collect()
    if BACKEND == 'cuda':
        if dev is not None and dev.type == 'cuda':
            with torch.cuda.device(dev): torch.cuda.empty_cache()
        else:
            for i in range(len(DEVICES)): 
                with torch.cuda.device(i): torch.cuda.empty_cache()
    elif BACKEND == 'mps':
        try: torch.mps.empty_cache()
        except Exception: pass

log(f"host={'kaggle' if ON_KAGGLE else platform.node()} backend={BACKEND} devices={DEVICES} vCPU={CPU}")
log(f"parallel models={MAX_PARALLEL} workers/model={NUM_WORKERS} (loader procs {MAX_PARALLEL*NUM_WORKERS}/{CPU})")
log(f"IMG_DIR={IMG_DIR}")
log(f"OUT_DIR={OUT_DIR} (preds -> {PRED_DIR})")
log(f"SIGNAL_GATE={SIGNAL_GATE} BAGGING={BAGGING} DROP_INCOMPLETE={DROP_INCOMPLETE} AMP={USE_AMP} mem={mem_str()}")

[00:00:00.0] host=Yashs-MacBook-Pro.local backend=mps devices=['mps'] vCPU=10
[00:00:00.0] parallel models=1 workers/model=4 (loader procs 4/10)
[00:00:00.0] IMG_DIR=./image_store
[00:00:00.0] OUT_DIR=. (preds -> ./preds)
[00:00:00.0] SIGNAL_GATE=False BAGGING=False DROP_INCOMPLETE=True AMP=False mem=rss 1.0G mps 0.0G


## 1. Market data

In [2]:
if not os.path.exists(PARQUET_PATH):
    raise FileNotFoundError(f"parquet not found: {PARQUET_PATH}")
df_market = pd.read_parquet(PARQUET_PATH)
df_market['Date'] = pd.to_datetime(df_market['Date'])

if TESTING:
    keep = df_market['Ticker'].unique()[:3]
    df_market = df_market[df_market['Ticker'].isin(keep)].copy()
    IMAGE_SIZES, HORIZONS, ENSEMBLE_SIZE = [5], [5], 2
    log("TESTING MODE: 3 tickers, 1 config, 2 members")

log(f"parquet: {len(df_market):,} rows | {df_market['Ticker'].nunique()} tickers | "
    f"{df_market['Date'].min().date()} -> {df_market['Date'].max().date()} | mem={mem_str()}")

[00:00:00.2] parquet: 2,739,387 rows | 495 tickers | 1997-12-31 -> 2023-12-29 | mem=rss 1.0G mps 0.0G


## 2. Image generation — exact PIL rendering (verified against the paper)

In [ ]:
def _height(n_days): return 32 if n_days == 5 else (64 if n_days == 20 else 96)
def img_shape(n_days): return _height(n_days), n_days * 3

def generate_image_uint8(window_df, n_days):
    height, width = img_shape(n_days)
    img = Image.new('L', (width, height), color=0)
    if len(window_df) == 0:
        return np.zeros((height, width), dtype=np.uint8)
    first_close = window_df['Close'].iloc[0]
    if first_close == 0 or np.isnan(first_close):
        return np.zeros((height, width), dtype=np.uint8)
    draw = ImageDraw.Draw(img)
    prices  = window_df[['Open','High','Low','Close']].values / first_close
    ma      = window_df['Close'].rolling(window=n_days, min_periods=1).mean().values / first_close
    volumes = window_df['Volume'].values
    max_p, min_p = np.nanmax(prices), np.nanmin(prices); max_v = np.nanmax(volumes)
    if max_p == min_p: max_p = min_p + 1e-5
    if max_v == 0:     max_v = 1e-5
    price_h = int(height * 0.8); vol_h = height - price_h
    def p2y(p): return int(np.clip(price_h-1-(p-min_p)/(max_p-min_p)*(price_h-1), 0, price_h-1))
    def v2y(v): return int(np.clip(height-1-(v/max_v)*(vol_h-1), price_h, height-1))
    ma_pts = []
    for i in range(len(window_df)):
        x = i*3 + 1
        oy, hy, ly, cy = p2y(prices[i,0]), p2y(prices[i,1]), p2y(prices[i,2]), p2y(prices[i,3])
        draw.line([(x,hy), (x,ly)], fill=255, width=1)
        draw.line([(x-1,oy), (x,oy)], fill=255, width=1)
        draw.line([(x,cy), (x+1,cy)], fill=255, width=1)
        draw.line([(x,height-1), (x,v2y(volumes[i]))], fill=255, width=1)
        ma_pts.append((x, p2y(ma[i])))
    if len(ma_pts) > 1:
        draw.line(ma_pts, fill=255, width=1)
    return np.array(img, dtype=np.uint8)

## 3. Image store — build (Mac) / load (Kaggle)

**The window-validity mask, fixed.** The old mask was
`t_df[REQUIRED].isna().rolling(n).sum().sum(axis=1) == 0`.
For the first `n-1` rows the rolling window is incomplete → every column is `NaN` → pandas' `sum(axis=1)` turns an all-`NaN` row into `0.0`, which **passes**. Those rows then sliced `iloc[idx-n+1 : idx+1]` with a *negative* start (`iloc[-19:1]`) → an **empty window** → an all-black image paired with a real label. Every ticker emitted `n-1` of them.

The fix reduces NaN to a per-day flag *first*, then rolls: an incomplete leading window stays `NaN`, and `NaN == 0` is `False`, so it is correctly excluded.

```python
bad_day    = t_df[COLS].isna().any(axis=1).astype(float)
window_bad = bad_day.rolling(n_days).sum()      # NaN on incomplete leading windows
valid      = np.where(window_bad.values == 0)[0]
```

`DROP_INCOMPLETE=True` *also* filters `end_idx < n_days-1` at load time — a no-op on a freshly built store, but it **rescues an old buggy dataset without regenerating**, since every corrupted sample is exactly that set.

In [ ]:
IMAGE_COLS  = ['Open','High','Low','Close','Volume']            # what the image needs (paper: CNN sees only this)
SIGNAL_COLS = ['MOM','STR','WSTR','TREND']                      # benchmarks, NOT CNN inputs

def _index_ticker(args):
    ticker, t_df, n_days, gate = args
    t_df = t_df.sort_values('Date').reset_index(drop=True)
    cols = IMAGE_COLS + (SIGNAL_COLS if gate else [])
    bad_day    = t_df[cols].isna().any(axis=1).astype(float)
    window_bad = bad_day.rolling(window=n_days).sum()            # NaN on the first n_days-1 rows
    valid      = np.where(window_bad.values == 0)[0]             # -> leading rows correctly excluded
    recs = [{'ticker': ticker, 'end_idx': int(i), 'end_date': str(t_df['Date'].iloc[i].date()),
             'MOM': float(t_df['MOM'].iloc[i]),  'STR':   float(t_df['STR'].iloc[i]),
             'WSTR': float(t_df['WSTR'].iloc[i]), 'TREND': float(t_df['TREND'].iloc[i])}
            for i in valid]
    return ticker, t_df, recs

_G = {}
def _render_chunk(rng):
    lo, hi = rng
    mm = np.memmap(_G['dat'], dtype=np.uint8, mode='r+', shape=_G['shape'])
    nd = _G['n_days']
    for i in range(lo, hi):
        s = _G['recs'][i]; tdf = _G['grouped'][s['ticker']]; e = s['end_idx']
        mm[i, 0] = generate_image_uint8(tdf.iloc[e-nd+1 : e+1], nd)
    mm.flush(); del mm
    return hi - lo

def store_paths(n_days):
    return (os.path.join(IMG_DIR, f'raw_n{n_days}.dat'),
            os.path.join(IMG_DIR, f'meta_n{n_days}.parquet'))

def build_store(df, n_days):
    """Render every valid window once into a raw uint8 memmap. Skips if already present."""
    dat, meta = store_paths(n_days)
    H, W = img_shape(n_days)
    if os.path.exists(dat) and os.path.exists(meta):
        log(f"N={n_days}: store already present, skipping generation.")
        return
    t0 = time.time()
    log(f"N={n_days}: generating images (gate_signals={SIGNAL_GATE})...")

    args = [(t, g.copy(), n_days, SIGNAL_GATE) for t, g in df.groupby('Ticker')]
    grouped, recs = {}, []
    with cf.ProcessPoolExecutor(max_workers=CPU) as ex:
        for tk, tdf, r in tqdm(ex.map(_index_ticker, args), total=len(args),
                               desc=f"index N={n_days}", leave=False):
            grouped[tk] = tdf; recs.extend(r)
    N = len(recs)
    if N == 0:
        log(f"N={n_days}: no valid windows."); return
    gb = N*H*W/1e9
    log(f"N={n_days}: {N:,} windows -> {gb:.1f} GB raw ({H}x{W} px). rendering on {CPU} cores...")

    np.memmap(dat, dtype=np.uint8, mode='w+', shape=(N,1,H,W)).flush()      # preallocate
    m = pd.DataFrame(recs)
    m.attrs = {}
    m.to_parquet(meta)

    _G.update(dat=dat, shape=(N,1,H,W), n_days=n_days, grouped=grouped, recs=recs)
    chunk  = max(1, N // (CPU*8))
    ranges = [(i, min(i+chunk, N)) for i in range(0, N, chunk)]
    with cf.ProcessPoolExecutor(max_workers=CPU) as ex:
        list(tqdm(ex.map(_render_chunk, ranges), total=len(ranges),
                  desc=f"render N={n_days}", leave=False))
    _G.clear(); del grouped, recs, m
    free_memory()
    log(f"N={n_days}: built in {(time.time()-t0)/60:.1f} min -> {dat} | mem={mem_str()}")

def load_store(n_days):
    """Memory-light: keeps ONLY the small metadata table + per-ticker Close arrays (never full DataFrames)."""
    dat, meta = store_paths(n_days)
    if not (os.path.exists(dat) and os.path.exists(meta)):
        return None
    m = pd.read_parquet(meta).reset_index(drop=True)
    H, W = img_shape(n_days)
    N = len(m)
    expect, actual = N*H*W, os.path.getsize(dat)
    if actual != expect:
        raise RuntimeError(f"N={n_days}: .dat is {actual:,} bytes but metadata implies {expect:,} "
                           f"({N:,} imgs of {H}x{W}). Wrong dataset, or built with different settings.")
    m['_row'] = np.arange(N)                                    # absolute row in the .dat -- fixed forever
    if DROP_INCOMPLETE:
        bad = m['end_idx'].values < (n_days - 1)
        if bad.any():
            log(f"N={n_days}: dropping {int(bad.sum()):,} blank rows from the old indexing bug "
                f"(end_idx < {n_days-1}); .dat left untouched.")
            m = m[~bad].reset_index(drop=True)
    log(f"N={n_days}: {N:,} images on disk ({actual/1e9:.1f} GB), {len(m):,} usable | mem={mem_str()}")
    return {'dat': dat, 'shape': (N,1,H,W), 'meta': m}

## 4. Labels, splits, dataset

Images are horizon-independent, so one store serves all three horizons — only the label changes. The **test set is seed-independent and chronological**, so it is identical for every ensemble member; that is what makes averaging their predictions column-wise valid (and what lets you train members on *different machines* and still combine them).

In [ ]:
def label_for_horizon(store, df, horizon):
    m = store['meta']
    closes = {t: g.sort_values('Date')['Close'].values.astype(np.float64)   # small: no DataFrames retained
              for t, g in df.groupby('Ticker')}
    fwd = np.full(len(m), np.nan)
    for t, pos in m.groupby('ticker').indices.items():
        c = closes.get(t)
        if c is None: continue
        ei = m['end_idx'].values[pos]
        ok = (ei + horizon) < len(c)
        base_p = c[ei]
        fwd_p  = np.where(ok, c[np.minimum(ei + horizon, len(c)-1)], np.nan)
        with np.errstate(invalid='ignore', divide='ignore'):
            fwd[pos] = np.where(ok & (base_p != 0), fwd_p/base_p - 1.0, np.nan)
    del closes

    d = m.copy(); d['Fwd_Ret'] = fwd
    d = d[~np.isnan(d['Fwd_Ret'].values)].reset_index(drop=True)   # drop the undefined-label tail
    rows  = d['_row'].values                                        # absolute rows into the .dat
    label = (d['Fwd_Ret'].values > 0).astype(np.int64)
    base  = d.rename(columns={'ticker':'Ticker', 'end_date':'Date'})[
                ['Date','Ticker','MOM','STR','WSTR','TREND','Fwd_Ret']]
    return rows, label, base

def trainval_positions(base):
    d = base['Date'].values
    return np.where((d >= '1997-01-01') & (d <= '2004-12-31'))[0]

def test_positions(base):
    d = base['Date'].values                                        # seed-independent AND chronological
    te = np.where((d >= '2005-01-01') & (d <= '2023-12-31'))[0]
    return te[np.argsort(d[te], kind='stable')]

def subsample_split(tv, seed, frac=TRAIN_FRAC):
    tv = tv.copy()
    np.random.default_rng(seed).shuffle(tv)
    cut = int(frac * len(tv))
    return tv[:cut], tv[cut:]

class ImgDataset(Dataset):
    def __init__(self, dat, shape, rows, labels=None):
        self.dat=dat; self.shape=tuple(int(x) for x in shape)
        self.rows=np.asarray(rows); self.labels=labels; self._mm=None
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        if self._mm is None:                                       # mapped lazily: one handle per worker
            self._mm = np.memmap(self.dat, dtype=np.uint8, mode='r', shape=self.shape)
        img = torch.from_numpy(np.ascontiguousarray(self._mm[int(self.rows[i])]))   # (1,H,W) uint8
        if self.labels is None: return img
        return img, int(self.labels[i])
    def close(self):
        self._mm = None                                            # release the mapping

## 5. CNN — verbatim

In [ ]:
def xavier_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)

class TrendCNN(nn.Module):
    def __init__(self, n_days=20):
        super().__init__()
        if n_days == 5:    blocks, vs, vd = 2, 1, 1
        elif n_days == 20: blocks, vs, vd = 3, 3, 2
        else:              blocks, vs, vd = 4, 3, 3
        layers, in_c, out_c = [], 1, 64
        for i in range(blocks):
            stride   = (vs, 1) if i == 0 else (1, 1)
            dilation = (vd, 1) if i == 0 else (1, 1)
            layers += [nn.Conv2d(in_c, out_c, (5,3), stride=stride, padding=(2,1), dilation=dilation),
                       nn.BatchNorm2d(out_c), nn.LeakyReLU(0.01), nn.MaxPool2d((2,1))]
            in_c, out_c = out_c, out_c*2
        self.features = nn.Sequential(*layers)
        H, W = img_shape(n_days)
        flat = self.features(torch.zeros(1,1,H,W)).view(1,-1).size(1)
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(flat, 2))
        self.apply(xavier_init)
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(x.size(0), -1))

## 6. Training engine — per-member checkpointing + explicit teardown

Every member writes `preds/pred_I{n}_R{h}_s{seed}.npy` on completion, so a member trained on the Mac and a member trained on Kaggle combine freely; re-running skips whatever already exists.

**Memory hygiene** (this is what stops the I60 creep): after each member the model, optimizer, all three DataLoaders and their persistent worker processes are dropped, the memmap handle is released, `gc.collect()` runs, and the device cache is emptied. RSS and GPU memory print every epoch — if they still climb monotonically, you'll see exactly where.

In [ ]:
def _mk_loader(ds, shuffle, device):
    kw = dict(batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
              pin_memory=(device.startswith('cuda')))
    if NUM_WORKERS > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    return DataLoader(ds, **kw)

def _shutdown(loader):
    """Persistent workers survive the loader unless the iterator is dropped first."""
    try:
        it = getattr(loader, '_iterator', None)
        if it is not None:
            it._shutdown_workers(); loader._iterator = None
    except Exception:
        pass

def pred_path(n_days, horizon, seed):
    return os.path.join(PRED_DIR, f'pred_I{n_days}_R{horizon}_s{seed}.npy')

def train_and_predict(job):
    n_days, horizon, dat, shape, tr, trl, va, val, te, device, seed, tag = job
    out = pred_path(n_days, horizon, seed)
    if os.path.exists(out):                                  # RESUME: this member is already done
        p = np.load(out)
        log(f"{tag} cached ({len(p):,} preds) -> skipping", tag='·')
        return p

    torch.manual_seed(seed); np.random.seed(seed)
    dev = torch.device(device)
    if dev.type == 'cuda': torch.cuda.set_device(dev)

    ds_tr = ImgDataset(dat, shape, tr, trl)
    ds_va = ImgDataset(dat, shape, va, val)
    ds_te = ImgDataset(dat, shape, te)
    l_tr = _mk_loader(ds_tr, True,  device)
    l_va = _mk_loader(ds_va, False, device)
    l_te = _mk_loader(ds_te, False, device)

    model     = TrendCNN(n_days).to(dev)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    amp    = USE_AMP and dev.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp)

    log(f"{tag} start on {device} | train {len(tr):,} val {len(va):,} test {len(te):,} | {mem_str(dev)}")
    best, patience_ctr, t_start = float('inf'), 0, time.time()
    ckpt = tempfile.NamedTemporaryFile(suffix='.pt', delete=False).name   # unique per member -> no races

    for epoch in range(EPOCHS):
        t0 = time.time(); model.train(); tr_loss = 0.0
        for imgs, y in l_tr:
            imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)    # uint8 -> float ON device
            y    = y.to(dev, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if amp:
                with torch.autocast('cuda'):
                    loss = criterion(model(imgs), y)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss = criterion(model(imgs), y)
                loss.backward(); optimizer.step()
            tr_loss += loss.item()
        tr_loss /= max(1, len(l_tr))

        model.eval(); va_loss = 0.0
        with torch.no_grad():
            for imgs, y in l_va:
                imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)
                y    = y.to(dev, non_blocking=True)
                if amp:
                    with torch.autocast('cuda'):
                        va_loss += criterion(model(imgs), y).item()
                else:
                    va_loss += criterion(model(imgs), y).item()
        va_loss /= max(1, len(l_va))

        mark = ''
        if va_loss < best:
            best, patience_ctr = va_loss, 0
            torch.save(model.state_dict(), ckpt); mark = 'best'
        else:
            patience_ctr += 1
            mark = f'patience {patience_ctr}/{PATIENCE}'
        log(f"{tag} ep{epoch+1:03d} train {tr_loss:.4f} val {va_loss:.4f} "
            f"{time.time()-t0:5.1f}s {mark} | {mem_str(dev)}")
        if patience_ctr >= PATIENCE:
            log(f"{tag} early stop at epoch {epoch+1} (best val {best:.4f})")
            break

    model.load_state_dict(torch.load(ckpt, map_location=dev, weights_only=True))
    os.remove(ckpt)

    model.eval(); probs = []
    with torch.no_grad():
        for imgs in l_te:
            imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)
            if amp:
                with torch.autocast('cuda'):
                    p = torch.softmax(model(imgs), 1)[:, 1]
            else:
                p = torch.softmax(model(imgs), 1)[:, 1]
            probs.append(p.float().cpu().numpy())
    probs = np.concatenate(probs) if probs else np.array([])
    np.save(out, probs)                                        # checkpoint: survives a crash / machine switch

    # ---- teardown: the anti-leak block ----
    for l in (l_tr, l_va, l_te): _shutdown(l)
    for d in (ds_tr, ds_va, ds_te): d.close()
    del l_tr, l_va, l_te, ds_tr, ds_va, ds_te, model, optimizer, criterion, scaler
    free_memory(dev)

    log(f"{tag} done in {(time.time()-t_start)/60:.1f} min | {len(probs):,} preds -> {os.path.basename(out)} "
        f"| {mem_str(dev)}")
    return probs

## 7. Portfolio evaluation — JKX Table 3

Rank names on the signal into 10 deciles each rebalance date, hold the equal-weight long-short (D10 − D1) for `H` days, and **rebalance every `H` trading days so periods never overlap** (5d ≈ weekly, 20d ≈ monthly, 60d ≈ quarterly). Equal-weighted only — the paper's value-weighted panel needs market cap, which this OHLCV parquet doesn't carry.

In [ ]:
def portfolio_performance(signal, df, horizon, min_names=20):
    d = df.dropna(subset=[signal, 'Fwd_Ret'])                 # benchmarks are NaN during their warm-up
    if d.empty:
        return dict(AvgHoldRet=np.nan, tstat=np.nan, AnnReturn=np.nan, AnnSharpe=np.nan, NPeriods=0)
    dates = np.sort(d['Date'].unique())
    d = d[d['Date'].isin(set(dates[::horizon]))]              # non-overlapping holding periods
    rets = []
    for _, g in d.groupby('Date'):
        if len(g) < min_names:                                # too thin to define 10 deciles
            continue
        dec = pd.qcut(g[signal].rank(method='first'), 10, labels=False)
        rets.append(g.loc[dec == 9, 'Fwd_Ret'].mean() - g.loc[dec == 0, 'Fwd_Ret'].mean())
    r = np.asarray(rets, float); r = r[~np.isnan(r)]; n = len(r)
    if n < 2:
        return dict(AvgHoldRet=(r.mean() if n else np.nan), tstat=np.nan,
                    AnnReturn=np.nan, AnnSharpe=np.nan, NPeriods=n)
    ppy = 252.0 / horizon
    mu, sd = r.mean(), r.std(ddof=1)
    return dict(AvgHoldRet=mu,
                tstat=(mu/(sd/np.sqrt(n)) if sd > 0 else np.nan),
                AnnReturn=mu*ppy,
                AnnSharpe=(mu/sd*np.sqrt(ppy) if sd > 0 else np.nan),
                NPeriods=n)

## 8. Run

Resumes automatically: configs already in `results.csv` are skipped, and within a config any member with a cached `.npy` is skipped. So you can stop this at any point, move machines, and re-run.

In [ ]:
# ---- generate any missing stores (no-op on Kaggle if the dataset is mounted) ----
for n_days in IMAGE_SIZES:
    dat, meta = store_paths(n_days)
    if os.path.exists(dat) and os.path.exists(meta):
        continue
    if ON_KAGGLE:
        log(f"N={n_days}: MISSING from IMG_DIR and generation on Kaggle is slow -- "
            f"generate on the Mac and upload. Skipping this size.")
        continue
    build_store(df_market, n_days)

# ---- resume: what's already finished? ----
done = set()
if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    done = set(prev['Config'].unique())
    log(f"resuming: {len(done)} configs already in {RESULTS_CSV} -> {sorted(done)}")
    results = prev.to_dict('records')
else:
    results = []

for n_days in IMAGE_SIZES:
    store = load_store(n_days)
    if store is None:
        log(f"N={n_days}: no store found, skipping."); continue

    for horizon in HORIZONS:
        cfg = f"I{n_days}/R{horizon}"
        if cfg in done:
            log(f"{cfg}: already complete, skipping."); continue
        log("=" * 66)
        log(f"CONFIG {cfg}  (ensemble {ENSEMBLE_SIZE}, {MAX_PARALLEL} concurrent on {BACKEND})")
        log("=" * 66)

        rows, label, base = label_for_horizon(store, df_market, horizon)
        te     = test_positions(base)
        tv_all = trainval_positions(base)
        if len(tv_all) == 0 or len(te) == 0:
            log(f"{cfg}: empty train or test split, skipping."); continue
        te_rows   = rows[te]
        eval_base = base.iloc[te].reset_index(drop=True)     # row-for-row aligned with every member's preds
        log(f"{cfg}: train/val pool {len(tv_all):,} | test {len(te):,} "
            f"| test dates {eval_base['Date'].min()} -> {eval_base['Date'].max()}")

        jobs = []
        for i in range(ENSEMBLE_SIZE):
            seed = 1000 + i
            tr, va = subsample_split(tv_all, seed if BAGGING else SHARED_SPLIT_SEED)
            jobs.append((n_days, horizon, store['dat'], store['shape'],
                         rows[tr], label[tr], rows[va], label[va], te_rows,
                         DEVICES[i % len(DEVICES)], seed, f"[{cfg} m{i+1}/{ENSEMBLE_SIZE}]"))

        preds = [None] * ENSEMBLE_SIZE
        t_cfg = time.time()
        if MAX_PARALLEL > 1:
            with cf.ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
                fut = {ex.submit(train_and_predict, j): k for k, j in enumerate(jobs)}
                for f in cf.as_completed(fut):
                    preds[fut[f]] = f.result()               # raises here if a member failed -- don't hide it
        else:
            for k, j in enumerate(jobs):
                preds[k] = train_and_predict(j)

        lens = {len(p) for p in preds}
        if lens != {len(te)}:
            raise RuntimeError(f"{cfg}: prediction lengths {lens} != test rows {len(te)}. "
                               f"Stale .npy from a different run? Delete {PRED_DIR} for this config.")

        eval_base['CNN_Prob'] = np.mean(preds, axis=0)
        log(f"{cfg}: ensemble complete in {(time.time()-t_cfg)/60:.1f} min | scoring...")

        for strat in ['CNN_Prob','MOM','STR','WSTR','TREND']:
            perf = portfolio_performance(strat, eval_base, horizon)
            results.append({'Config': cfg, 'Strategy': strat, **perf})
            log(f"  {strat:9s} AnnRet {perf['AnnReturn']:+.4f}  Sharpe {perf['AnnSharpe']:+.3f}  "
                f"t {perf['tstat']:+.2f}  periods {perf['NPeriods']}")

        pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)    # checkpoint after every config
        del preds, eval_base, rows, label, base, te, tv_all, te_rows
        free_memory()
        log(f"{cfg}: saved -> {RESULTS_CSV} | mem={mem_str()}")

    del store
    free_memory()

df_results = pd.DataFrame(results)
log(f"ALL DONE in {(time.time()-_T0)/60:.1f} min | mem={mem_str()}")
display(df_results)

## 9. Results

In [ ]:
df_results = pd.read_csv(RESULTS_CSV)
piv = df_results.pivot(index='Config', columns='Strategy', values='AnnSharpe')
order = [f'I{i}/R{h}' for i in [5,20,60] for h in [5,20,60] if f'I{i}/R{h}' in piv.index]
print("Annualized Sharpe (equal-weight decile long-short, non-overlapping):")
display(piv.loc[order][['CNN_Prob','MOM','STR','WSTR','TREND']].round(3))
print("\nCNN detail:")
display(df_results[df_results.Strategy=='CNN_Prob']
        .set_index('Config')[['AnnReturn','AnnSharpe','tstat','NPeriods']].round(3))